# 3. Properties (Getters/Setters)

In Java, you write getters and setters from day one because switching from a field to a method is a **breaking change**. Python's `@property` solves this beautifully — start with a plain attribute, and add validation later **without changing any client code**.

This notebook covers: `@property`, `@setter`, `@deleter`, computed properties, `@cached_property`, and the Pythonic philosophy of attribute access.

### 3.1 Why Properties? The Java Problem

**☕ JAVA:** You must write getters/setters upfront, because changing `person.age` to `person.getAge()` later breaks all client code:
```java
// Start with a field:
public int age;           // client: person.age = 5

// Later you need validation — BREAKING CHANGE:
private int age;
public int getAge() { return age; }
public void setAge(int age) {
    if (age < 0) throw new IllegalArgumentException();
    this.age = age;         // client must change to: person.setAge(5)
}
```

**🐍 PYTHON:** No problem! Start with a plain attribute. If you need validation later, add `@property` — **client code stays exactly the same** (`person.age = 5` still works).

In [ ]:
# STEP 1: Start simple — plain attribute
class PersonV1:
    def __init__(self, name: str, age: int):
        self.name = name
        self.age = age       # Just a plain attribute

p = PersonV1("Alice", 30)
p.age = 31                   # Direct access
print(f"{p.name}: {p.age}")
p.age = -5                   # 😱 No validation! Bug goes unnoticed
print(f"Oops: {p.age}")

### 3.2 Adding a Getter with `@property`

**☕ JAVA:** `public int getAge() { return this.age; }`

**🐍 PYTHON:** `@property` turns a method into an attribute-like access. Store the real value in a `_private` variable.

In [ ]:
class User:
    def __init__(self, name: str, age: int):
        self.name = name
        self._age = age       # Convention: underscore for "private" backing field

    @property
    def age(self) -> int:
        """Getter — accessed like an attribute, not a method call."""
        return self._age

user = User("Alice", 30)
print(f"Age: {user.age}")     # Looks like attribute access, but calls the method!
# print(user.age())           # ❌ TypeError — it's not a method call!

### 3.3 Adding a Setter with Validation

**☕ JAVA:**
```java
public void setAge(int age) {
    if (age < 0) throw new IllegalArgumentException("Age cannot be negative");
    this.age = age;
}
```

**🐍 PYTHON:** Use `@property_name.setter`. The client still writes `user.age = 31` — same syntax as a plain attribute!

In [ ]:
# STEP 2: Add validation — NO client code changes!
class PersonV2:
    def __init__(self, name: str, age: int):
        self.name = name
        self.age = age        # This now goes through the setter!

    @property
    def age(self) -> int:
        return self._age

    @age.setter
    def age(self, value: int) -> None:
        if value < 0:
            raise ValueError(f"Age cannot be negative, got {value}")
        if value > 150:
            raise ValueError(f"Age unrealistic, got {value}")
        self._age = value

p = PersonV2("Alice", 30)
p.age = 31                    # ✅ Same syntax as PersonV1 — goes through setter
print(f"{p.name}: {p.age}")

try:
    p.age = -5                # ❌ Now caught!
except ValueError as e:
    print(f"Validation: {e}")

> 💡 **Key insight:** Notice that `self.age = age` in `__init__` also goes through the setter — giving you validation at construction time too!

### 3.4 Computed (Read-Only) Properties

**☕ JAVA:** Computed values are always methods: `rectangle.getArea()`

**🐍 PYTHON:** A `@property` without a setter is automatically **read-only**. Great for values derived from other attributes.

In [ ]:
class Rectangle:
    def __init__(self, width: float, height: float):
        self.width = width
        self.height = height

    @property
    def area(self) -> float:
        """Computed — recalculated every time it's accessed."""
        return self.width * self.height

    @property
    def perimeter(self) -> float:
        return 2 * (self.width + self.height)

    @property
    def is_square(self) -> bool:
        return self.width == self.height

rect = Rectangle(10, 5)
print(f"Area:      {rect.area}")       # Looks like an attribute!
print(f"Perimeter: {rect.perimeter}")
print(f"Is square: {rect.is_square}")

try:
    rect.area = 100                     # ❌ Read-only — no setter defined!
except AttributeError as e:
    print(f"Blocked: {e}")

### 3.5 The `@deleter` — Optional Cleanup

**☕ JAVA:** No equivalent — fields can't be "deleted".

**🐍 PYTHON:** `@property_name.deleter` lets you hook into `del obj.attr`. Rarely used, but useful for cleanup or resetting state.

In [ ]:
class CachedProfile:
    def __init__(self, username: str):
        self.username = username
        self._avatar = None

    @property
    def avatar(self) -> str:
        if self._avatar is None:
            # Simulate expensive computation
            self._avatar = f"avatar_{self.username}.png"
            print("  (generated avatar)")
        return self._avatar

    @avatar.deleter
    def avatar(self):
        """Clear the cache — next access will regenerate."""
        print("  (cache cleared)")
        self._avatar = None

profile = CachedProfile("alice")
print(f"Avatar: {profile.avatar}")    # Generates
print(f"Avatar: {profile.avatar}")    # Cached — no regeneration
del profile.avatar                     # Clear cache
print(f"Avatar: {profile.avatar}")    # Regenerates

### 3.6 `@cached_property` — Built-in Lazy Caching (Python 3.8+)

The manual caching pattern in 3.5 is so common that Python provides `functools.cached_property`. It computes the value **once**, then stores it as a regular attribute — no repeated computation.

| Feature | `@property` | `@cached_property` |
|---------|------------|--------------------|
| Recalculated on each access | ✅ Yes | ❌ No — computed once |
| Supports setter | ✅ Yes | ❌ No |
| Thread-safe | N/A | ✅ Yes (Python 3.12+) |
| Clear cache | Via `@deleter` | `del obj.attr` |

In [ ]:
from functools import cached_property
import time

class DataAnalysis:
    def __init__(self, data: list[int]):
        self.data = data

    @cached_property
    def summary(self) -> dict:
        """Expensive computation — only runs ONCE."""
        print("  (computing summary...)")
        time.sleep(0.1)   # Simulate expensive work
        return {
            "mean": sum(self.data) / len(self.data),
            "min": min(self.data),
            "max": max(self.data),
            "count": len(self.data),
        }

analysis = DataAnalysis([10, 20, 30, 40, 50])
print(f"First access:  {analysis.summary}")    # Computes
print(f"Second access: {analysis.summary}")    # Cached — instant!

# To invalidate the cache, delete the attribute:
del analysis.summary
print(f"After clear:   {analysis.summary}")    # Recomputes

> ⚠️ **Important:** `@cached_property` only works on instances that have a `__dict__` (i.e., no `__slots__`). It stores the result directly in the instance dictionary.

### 3.7 How Properties Work — The Descriptor Protocol

**☕ JAVA:** Annotations like `@Override` are compile-time metadata — they don't change how attribute access works.

**🐍 PYTHON:** `@property` is actually a **descriptor** — a class that implements `__get__`, `__set__`, and `__delete__`. When Python sees `obj.age`, it checks if `age` in the class is a descriptor and calls its methods. This is not magic — it's a protocol you can implement yourself!

In [ ]:
# What @property actually does under the hood:
class Positive:
    """A reusable descriptor that enforces positive values."""

    def __init__(self, name: str):
        self.name = name
        self.private_name = f"_{name}"

    def __get__(self, obj, objtype=None):
        if obj is None: return self
        return getattr(obj, self.private_name)

    def __set__(self, obj, value):
        if value <= 0:
            raise ValueError(f"{self.name} must be positive, got {value}")
        setattr(obj, self.private_name, value)

class Product:
    # Reusable validators — no repeated property code!
    price = Positive("price")
    quantity = Positive("quantity")

    def __init__(self, name: str, price: float, quantity: int):
        self.name = name
        self.price = price         # Goes through Positive.__set__
        self.quantity = quantity

p = Product("Widget", 9.99, 100)
print(f"{p.name}: ${p.price} × {p.quantity}")

try:
    p.price = -5   # Caught by descriptor!
except ValueError as e:
    print(f"Descriptor: {e}")

> 💡 **When to use descriptors over `@property`:** When you have the **same validation logic** on multiple attributes. Instead of writing 3 identical properties, write one descriptor and reuse it.

### 3.8 Write-Only Properties — A Rare Pattern

Python allows a setter without a meaningful getter — useful for security-sensitive values like passwords where reading back the raw value should be blocked.

In [ ]:
import hashlib

class Account:
    def __init__(self, username: str, password: str):
        self.username = username
        self.password = password    # Goes through setter

    @property
    def password(self) -> str:
        """Getter intentionally blocks reading the raw password."""
        raise AttributeError("Password is write-only!")

    @password.setter
    def password(self, value: str) -> None:
        if len(value) < 8:
            raise ValueError("Password must be at least 8 characters")
        self._password_hash = hashlib.sha256(value.encode()).hexdigest()

    def verify(self, password: str) -> bool:
        return self._password_hash == hashlib.sha256(password.encode()).hexdigest()

acc = Account("alice", "securepass123")
print(f"Verify correct:  {acc.verify('securepass123')}")
print(f"Verify wrong:    {acc.verify('wrong')}")

try:
    print(acc.password)   # ❌ Blocked!
except AttributeError as e:
    print(f"Read blocked: {e}")

---

## 🧪 Try It Yourself

**Exercise 1:** Create a `Temperature` class with a `celsius` property (getter/setter). Add a read-only `fahrenheit` property computed as `C × 9/5 + 32`. The setter should reject values below -273.15 (absolute zero).

In [ ]:
# Exercise 1: Your code here


**Exercise 2:** Create a `Password` class with a `password` property. The setter should:
- Reject passwords shorter than 8 characters
- Store the password as a hash (use `hash()` for simplicity)
- The getter should return `"****"` (never reveal the actual password)

In [ ]:
# Exercise 2: Your code here


**Exercise 3:** Create a `Circle` class with a `radius` property (rejects negative values). Add computed read-only properties for `area` and `circumference`. Use `@cached_property` for `area` and show how modifying `radius` requires invalidating the cache.

In [ ]:
# Exercise 3: Your code here


---

## 📝 Key Takeaways: Java → Python

| Concept | Java | Python |
|---------|------|--------|
| Getter | `getAge()` | `@property` |
| Setter | `setAge(value)` | `@age.setter` |
| Deleter | Not possible | `@age.deleter` |
| Client syntax | `obj.getAge()` / `obj.setAge(5)` | `obj.age` / `obj.age = 5` |
| Read-only | `private` field + getter only | `@property` without setter |
| Write-only | Not common | Getter raises `AttributeError` |
| Computed value | `getArea()` method | `@property` — looks like an attribute |
| Lazy caching | Manual field + null check | `@cached_property` (Python 3.8+) |
| Add validation later | Breaks all client code | Seamless — no changes needed |
| Backing field | `private int age;` | `self._age` (convention) |
| Reusable validation | Custom annotation processor | Descriptor protocol |
| Best practice | Always write getters/setters | Start with plain attributes |